[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eo-agh/data-analysis-earth-sciences/blob/main/docs/zarr.ipynb)

## Zarr

Zarr to nowoczesny format przechowywania **wielowymiarowych tablic (n-d arrays)**, zaprojektowany z myślą o środowiskach chmurowych i dużych zbiorach danych naukowych. W odróżnieniu od tradycyjnych formatów rastrowych (GeoTIFF) czy naukowych (NetCDF/HDF5), Zarr nie trzyma danych w jednym monolitycznym pliku - zamiast tego dzieli tablicę na **chunki (fragmenty)**, które są zapisywane jako osobne obiekty w dowolnym systemie przechowywania: lokalnym systemie plików, Amazon S3, Google Cloud Storage czy Azure Blob Storage.

Kluczowe właściwości:

- **Chunking** - tablica dzielona jest na regularne bloki o ustalonym rozmiarze; dostęp do danych wymaga pobrania tylko niezbędnych chunków, nie całego pliku.
- **Kompresja na poziomie chunka** - każdy fragment jest niezależnie kompresowany (np. Blosc+Zstd, Zlib, LZ4), co obniża koszty transferu i składowania.
- **Równoległy dostęp** - niezależne chunki można czytać i zapisywać jednocześnie, świetnie integrując się z biblioteką Dask.
- **Hierarchia grup** - dane organizowane są w drzewiaste struktury *grup* i *tablic* (analogicznie do katalogów i plików).
- **Metadane JSON** - każda tablica i grupa posiada czytelny dla człowieka plik `.zarray` / `.zgroup` opisujący kształt, dtype i układ chunków.

Aktualna wersja to **Zarr v3** (od początku 2025 r.), która m.in. ujednoliciła system kodeków (zastąpiła `compressor` i `filters` jednym polem `codecs`) i dodała natywne wsparcie dla nazw wymiarów - co ułatwia integrację z modelem danych NetCDF/CF.

### Porównanie z GeoTIFF i NetCDF

| Cecha | GeoTIFF (COG) | NetCDF / HDF5 | Zarr |
|---|---|---|---|
| Struktura | Jeden plik | Jeden plik | Wiele obiektów (katalog lub S3 prefix) |
| Chunking | Wewnętrzne tiling | Wewnętrzne chunking | Zewnętrzne, konfigurowalne |
| Równoległy zapis | Ograniczony | Ograniczony | Natywny |
| Cloud-native | Tak (COG) | Nie (wymaga seekowania) | Tak |
| CRS / geo-metadata | Natywny | Przez konwencje CF | Przez GeoZarr / rioxarray |

In [ ]:
# pip install zarr numpy xarray rioxarray matplotlib pandas
import zarr
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt

### Tworzenie zarr store - podstawowe operacje

Poniżej tworzymy prosty lokalny zarr store zawierający tablicę 3D reprezentującą serię czasową danych rastrowych `(czas, wiersze, kolumny)`.

In [ ]:
store = zarr.open("example.zarr", mode="w")

# 12 miesięcy × 256 wierszy × 256 kolumn, chunkowane po jednym miesiącu
ndvi = store.create_dataset(
    "ndvi",
    shape=(12, 256, 256),
    chunks=(1, 64, 64),
    dtype="float32",
    compressor=zarr.Blosc(cname="zstd", clevel=3),
)

ndvi[:] = np.random.uniform(-0.2, 0.9, size=(12, 256, 256)).astype("float32")
ndvi.attrs["long_name"] = "Normalized Difference Vegetation Index"
ndvi.attrs["units"] = "-"

print(store.tree())
print(f"\nRozmiar chunka : {ndvi.chunks}")
print(f"Liczba chunków : {ndvi.nchunks}")
print(f"Rozmiar na dysku: {ndvi.nbytes_stored / 1024:.1f} kB  (oryginał: {ndvi.nbytes / 1024:.0f} kB)")

### Zarr + xarray

W praktyce Zarr najczęściej używamy przez warstwę **xarray**, która obsługuje metadane, współrzędne i konwencje CF. xarray automatycznie dzieli dane na chunki Daska podczas odczytu dużych store'ów z chmury.

In [ ]:
times = pd.date_range("2023-01-01", periods=12, freq="MS")
lat   = np.linspace(49.0, 50.0, 256)
lon   = np.linspace(19.0, 20.0, 256)

ds = xr.Dataset(
    {
        "ndvi": (("time", "lat", "lon"),
                 np.random.uniform(-0.2, 0.9, (12, 256, 256)).astype("float32")),
        "lst":  (("time", "lat", "lon"),
                 np.random.uniform(5, 35, (12, 256, 256)).astype("float32")),
    },
    coords={"time": times, "lat": lat, "lon": lon},
    attrs={"description": "Syntetyczny dataset dla okolic Krakowa"},
)

# Zapis do Zarr - tworzy katalog my_dataset.zarr/
ds.to_zarr("my_dataset.zarr", mode="w")
print("Zapisano do my_dataset.zarr")

# Odczyt - dane nie są od razu ładowane do pamięci (lazy loading)
ds_loaded = xr.open_zarr("my_dataset.zarr")
print(ds_loaded)

In [ ]:
ds_loaded["ndvi"].isel(time=6).plot(cmap="RdYlGn", vmin=-0.2, vmax=0.9, figsize=(7, 5))
plt.title("NDVI - lipiec 2023 (dane syntetyczne)")
plt.show()

## GeoZarr

**GeoZarr** to specyfikacja OGC (Open Geospatial Consortium) rozszerzająca Zarr o pełną semantykę geoprzestrzenną. Definiuje, jak wewnątrz struktury Zarr należy przechowywać:

- **Układ współrzędnych (CRS)** - np. WGS84 (EPSG:4326) lub UTM, zapisany jako WKT lub kod EPSG w atrybutach grupy,
- **Georeferencing** - transformację afiniczną mapującą piksele na współrzędne geograficzne (analogicznie do GeoTIFF),
- **Wieloskalowe przeglądy** (*multiscale overviews*) - piramidy rozdzielczości umożliwiające szybkie przeglądanie na małych skalach,
- **Konwencje CF** - standardowe atrybuty meteorologiczne i klimatyczne (`units`, `standard_name`, `grid_mapping`).

Dzięki GeoZarr dane rastrowe mogą być składowane w chmurze z pełnym kontekstem przestrzennym - bez konieczności konwersji na GeoTIFF. Biblioteki `rioxarray` i `GDAL >= 3.4` obsługują odczyt takich store'ów.

### Kerchunk - wirtualny zarr store

Pokrewnym podejściem jest **Kerchunk**: biblioteka generująca pliki referencyjne (JSON), które "udają" zarr store z punktu widzenia xarray, a fizycznie wskazują na zakresy bajtów w oryginalnych plikach NetCDF/HDF5/GRIB2 przechowywanych w chmurze. Nie wymaga przepisywania danych - wystarczy wygenerować mały plik referencyjny raz, a następnie korzystać z szybkiego dostępu chunkami.

### Zapis z informacją o CRS przez rioxarray

In [ ]:
import rioxarray  # rejestruje akcesor .rio na xarray DataArray/Dataset

ndvi_da = ds["ndvi"].rio.write_crs("EPSG:4326")

# Zapis DataArray z CRS do Zarr
ndvi_da.to_dataset(name="ndvi").to_zarr("my_geo_dataset.zarr", mode="w")

# Odczyt i weryfikacja CRS
ds_geo = xr.open_zarr("my_geo_dataset.zarr")
print("CRS:", ds_geo["ndvi"].rio.crs)

### Otwieranie zarr store z chmury

Jedną z największych zalet Zarr jest możliwość pracy z danymi bezpośrednio w chmurze - pobieramy tylko potrzebne chunki. Poniżej przykład otwarcia publicznie dostępnego **ARCO ERA5** (Analysis-Ready Cloud-Optimized ERA5) na Google Cloud Storage - globalnych danych reanaliz atmosferycznych w formacie Zarr.

Analogiczne store'y są dostępne na AWS (np. ERA5 na S3 `s3://era5-pds/zarr/`) i w Microsoft Planetary Computer.

In [ ]:
# pip install gcsfs
import gcsfs

gcs = gcsfs.GCSFileSystem(token="anon")
store_url = "gs://gcp-public-data-arco-era5/ar/1959-2022-full_37-1h-0p25deg-chunk-1.zarr-v3"

ds_era5 = xr.open_zarr(
    gcsfs.GCSMap(store_url, gcs=gcs),
    consolidated=True,
    chunks="auto",  # automatyczne chunki Dask
)
print(ds_era5)

# Wszystkie operacje są lazy - dane nie są pobierane do pamięci
# Dopiero .compute() lub wizualizacja wyzwala faktyczny transfer
t2m_slice = ds_era5["2m_temperature"].sel(
    time="2020-07-01T12:00",
    latitude=slice(55, 49),
    longitude=slice(14, 24),
)
t2m_slice.compute().plot(figsize=(8, 5), cmap="RdBu_r")
plt.title("ERA5: temperatura 2 m - Polska, 2020-07-01 12:00 UTC")
plt.show()